# ASVspoof 5 Track 1 - Colab 학습 노트북 (full train)

WavLM-Large(frozen) + adapter + multi-task workers + LCNN/SENet 구조를 Colab에서 학습한다.
순서: smoke test -> sanity probe -> 본 학습 (A1×C1: BiLSTM + LCNN-big) -> dev 평가 (`--metric official`).

사전 준비
- 코드는 Drive `MyDrive/STUDY/Project/ASVspoof5/`에 둔다 (`train.py`, `evaluate.py`, `configs/`, `data/`, `models/`, `scripts/`, `utils/`).
- 데이터셋이 gated일 수 있으므로 다운로드 전에 `huggingface-cli login` 또는 `HF_TOKEN` 설정.
- GPU 런타임 필요 (T4 이상, VRAM 15 GB 이상 권장).

대략적인 소요 시간 (T4 기준)

| 단계 | 시간 |
|---|---|
| 준비 + 의존성 | 약 5분 |
| 데이터 다운로드 | 30–60분 (54 GB) |
| tar 추출 | 8–15분 (매 세션) |
| smoke test | 5분 |
| sanity probe | 10–20분 |
| 본 학습 (train 182K) | epoch당 약 3.5시간 |
| dev 평가 | 1–2시간 |

train 셋이 커서 Colab Free 12시간 세션 한 번으로는 여러 epoch을 돌리기 어렵다. 끊기면 resume으로 이어서 학습한다. 시간이 부족하면 `ASVspoof5_colab_subset.ipynb`(50% subset)를 쓴다.

## 셀 2 - GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Runtime > Change runtime type > GPU 선택 후 다시 실행하세요.")
p = torch.cuda.get_device_properties(0)
print(f"GPU       : {p.name}")
print(f"VRAM      : {p.total_memory / 1024**3:.1f} GB")
print(f"PyTorch   : {torch.__version__}")
if p.total_memory / 1024**3 < 14.0:
    print("WavLM-Large frozen+batch 8은 약 11GB 사용. 15GB 미만 GPU는 batch 4로 자동 축소됨.")

## 셀 3 - Google Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 셀 4 - 경로 / 환경 변수
`DRIVE_ROOT`만 본인 Drive 경로로 바꾸면 된다. 데이터와 ckpt는 `/content`(휘발)에 두고 ckpt만 Drive로 백업한다.

In [ ]:
import os

# 사용자 설정
GITHUB_REPO = "https://github.com/Eunchae-L/ASVspoof.git"   # (참고용) 코드는 Drive에 영구 보관됨
DRIVE_ROOT  = "/content/drive/MyDrive/STUDY/Project/ASVspoof5"

PROJECT_ROOT = DRIVE_ROOT                     # 코드 위치 (Drive, 영구 - 모든 .py/configs/scripts 여기 있음)
DATA_ROOT    = "/content/data/asvspoof5"      # 추출된 flac/protocol (세션 종료 시 사라짐)
RAW_DIR      = "/content/raw"                 # 다운받은 tar (휘발! ~54GB - Drive 절약, 세션마다 재다운로드)
HF_HOME      = f"{DRIVE_ROOT}/.hf_cache"      # WavLM 모델 캐시 (영구, ~1.3 GB)
CKPT_DIR     = "/content/ckpt"                # 학습 중 ckpt (휘발, 빠른 디스크)
CKPT_BACKUP  = f"{DRIVE_ROOT}/ckpt"           # ckpt 백업 (영구)

os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HOME

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(CKPT_BACKUP, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}   (=DRIVE_ROOT, 코드 위치)")
print(f"RAW_DIR      = {RAW_DIR}   (휘발 - Drive 절약, 세션마다 재다운로드)")
print(f"DATA_ROOT    = {DATA_ROOT}  (세션마다 추출)")
print(f"HF_HOME      = {HF_HOME}")
print(f"CKPT_DIR     = {CKPT_DIR}   (휘발)")
print(f"CKPT_BACKUP  = {CKPT_BACKUP}   (영구)")

## 셀 5 - 작업 디렉터리 이동
코드는 Drive에 있으므로 clone 없이 cd만 한다.

In [ ]:
import os

# 코드는 Drive에 영구 보관됨. cwd만 PROJECT_ROOT로 이동.
os.chdir(PROJECT_ROOT)
print(f"cwd = {os.getcwd()}")
!ls -la

# (선택) GitHub에서 최신 변경사항을 Drive로 pull 받고 싶다면 아래 주석 해제:
# if os.path.isdir(f"{PROJECT_ROOT}/.git"):
#     !cd $PROJECT_ROOT && git pull --ff-only

## 셀 6 - 의존성 설치
Colab에 기본으로 없는 패키지만 설치.

In [ ]:
!pip install -q "transformers>=4.40" "huggingface_hub>=0.23" librosa soundfile pyyaml pandas

import transformers, huggingface_hub, librosa, soundfile, yaml, torch, torchaudio, sklearn, scipy, numpy, pandas
print(f"transformers     {transformers.__version__}")
print(f"huggingface_hub  {huggingface_hub.__version__}")
print(f"librosa          {librosa.__version__}")
print(f"soundfile        {soundfile.__version__}")
print(f"pandas           {pandas.__version__}")
print(f"pyyaml           {yaml.__version__}")
print(f"torch            {torch.__version__}")
print(f"torchaudio       {torchaudio.__version__}")
print(f"scikit-learn     {sklearn.__version__}")
print(f"scipy            {scipy.__version__}")
print(f"numpy            {numpy.__version__}")

## 셀 7 - 데이터 다운로드
HF에서 tar를 `RAW_DIR`로 받는다 (약 54 GB). 이미 있으면 skip.

In [ ]:
import os, glob

# 이미 받아져 있으면 skip
have_protocols = bool(glob.glob(f"{RAW_DIR}/ASVspoof5_protocols.tar*"))
have_train     = len(glob.glob(f"{RAW_DIR}/flac_T_*.tar")) > 0
have_dev       = len(glob.glob(f"{RAW_DIR}/flac_D_*.tar")) > 0

if have_protocols and have_train and have_dev:
    print("[download] 이미 Drive에 받아져 있음 → skip")
    !ls -la $RAW_DIR
else:
    print("[download] HF에서 받기 시작 (~54 GB)")
    !python scripts/download_asvspoof5.py --local-dir $RAW_DIR
    !ls -la $RAW_DIR

## 셀 8 - tar 추출
`/content/data/asvspoof5/`에 푼다. 매 세션 필요, 약 8–15분.

In [ ]:
import os, time, glob, shutil, subprocess

# 1. 디스크 사전 점검
def disk_free_gb(path="/content"):
    total, used, free = shutil.disk_usage(path)
    return used/1024**3, total/1024**3, free/1024**3

used, total, free = disk_free_gb("/content")
print(f"[disk] /content : {used:.1f}/{total:.1f} GB used, {free:.1f} GB free")
if free < 85:
    print(f"/content 여유공간이 85GB 미만입니다. ASVspoof 5 전체 추출은 ~83GB가 필요합니다.")
    print(f"   대안: train만 추출하거나, 일부 shard만 처리하세요.")

# 2. RAW_DIR 점검
print(f"\n[check] RAW_DIR = {RAW_DIR}")
if not os.path.isdir(RAW_DIR):
    raise FileNotFoundError(f"RAW_DIR이 존재하지 않음: {RAW_DIR}")
for entry in sorted(os.listdir(RAW_DIR)):
    full = os.path.join(RAW_DIR, entry)
    if os.path.isfile(full):
        sz = os.path.getsize(full) / 1024**3
        print(f"  {entry:<45} {sz:>7.2f} GB")
    else:
        print(f"  {entry}/")

raw_protocols = sorted(glob.glob(f"{RAW_DIR}/**/ASVspoof5_protocols.tar*", recursive=True))
raw_train     = sorted(glob.glob(f"{RAW_DIR}/**/flac_T_*.tar*", recursive=True))
raw_dev       = sorted(glob.glob(f"{RAW_DIR}/**/flac_D_*.tar*", recursive=True))
print(f"\n[check] protocols: {len(raw_protocols)}  flac_T: {len(raw_train)}  flac_D: {len(raw_dev)}")
if not raw_protocols or not raw_train or not raw_dev:
    raise FileNotFoundError("필요한 tar가 RAW_DIR에 없습니다.")

# 3. 이미 추출되었는지
def find_file(name, root):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

def find_dir(name, root):
    for r, dirs, _ in os.walk(root):
        if name in dirs:
            return os.path.join(r, name)
    return None

train_tsv  = find_file("ASVspoof5.train.tsv", DATA_ROOT)
dev_tsv    = find_file("ASVspoof5.dev.track_1.tsv", DATA_ROOT)
flac_T_dir = find_dir("flac_T", DATA_ROOT)
flac_D_dir = find_dir("flac_D", DATA_ROOT)

if train_tsv and dev_tsv and flac_T_dir and flac_D_dir:
    print(f"\n[extract] 이미 추출되어 있음 → skip")
else:
    # 4. 추출 (verbose error capture)
    t0 = time.time()
    os.makedirs(DATA_ROOT, exist_ok=True)

    def extract(tar_path, label):
        flag = "-xzf" if tar_path.endswith(".gz") else "-xf"
        before_free = disk_free_gb("/content")[2]
        r = subprocess.run(["tar", flag, tar_path, "-C", DATA_ROOT],
                           capture_output=True, text=True, check=False)
        after_free = disk_free_gb("/content")[2]
        delta = before_free - after_free
        if r.returncode != 0:
            print(f"\ntar 실패 (exit {r.returncode}) - {os.path.basename(tar_path)}")
            print(f"   stderr (앞 800자):\n{r.stderr[:800]}")
            print(f"\n[disk] 실패 시점: {after_free:.1f} GB free  (이번 tar로 {delta:.1f} GB 소모)")
            if "No space left" in (r.stderr or "") or after_free < 1.0:
                print("\n디스크 부족이 원인입니다.")
                print("   해결책 (택일):")
                print("   1) Colab Pro로 더 큰 디스크 확보")
                print("   2) 일부 shard만 사용: aborted된 위치 이전까지의 데이터로 학습")
                print("      → 이미 푼 flac은 살아있으니 셀 8은 skip하고 셀 10(config)으로 이동")
                print("   3) dev shard만 또는 train shard 일부만 풀기 (코드 수정 필요)")
            raise RuntimeError(f"tar failed on {label}")
        print(f"  {os.path.basename(tar_path):<25} ok  (사용 {delta:>5.1f} GB, 남은 {after_free:>5.1f} GB)")

    print(f"\n[extract] protocols")
    extract(raw_protocols[0], "protocols")

    print(f"\n[extract] flac_T ({len(raw_train)} shards)")
    for t in raw_train:
        extract(t, "flac_T")

    print(f"\n[extract] flac_D ({len(raw_dev)} shards)")
    for t in raw_dev:
        extract(t, "flac_D")

    print(f"\n[extract] done in {time.time()-t0:.0f}s")

    train_tsv  = find_file("ASVspoof5.train.tsv", DATA_ROOT)
    dev_tsv    = find_file("ASVspoof5.dev.track_1.tsv", DATA_ROOT)

# 5. 결과 검증
used, total, free = disk_free_gb("/content")
print(f"\n[disk] /content : {used:.1f}/{total:.1f} GB used, {free:.1f} GB free")
print(f"[check] DATA_ROOT layout:")
for line in subprocess.check_output(["ls", "-la", DATA_ROOT]).decode().splitlines():
    print(f"  {line}")

if not (train_tsv and dev_tsv):
    raise FileNotFoundError(f"protocol tsv 못 찾음. train={train_tsv}, dev={dev_tsv}")

def wc(path):
    return int(subprocess.check_output(["wc", "-l", path]).split()[0])

n_train = wc(train_tsv)
n_dev   = wc(dev_tsv)
print(f"\ntrain protocol rows: {n_train}  ({train_tsv})")
print(f"dev   protocol rows: {n_dev}  ({dev_tsv})")
assert n_train > 180_000, "train protocol row count too low"
assert n_dev   > 140_000, "dev protocol row count too low"

expected_root = os.path.dirname(os.path.dirname(train_tsv))
if expected_root != DATA_ROOT.rstrip("/"):
    print(f"\n실제 데이터 루트는 {expected_root} 입니다 (DATA_ROOT 와 다름).")
    print(f"    cfg['data']['root'] = '{expected_root}' 로 셀 10에서 덮어쓰세요.")

## 셀 9 - ckpt 디렉터리
`cfg["data"]["root"]`에 절대경로를 넣으므로 symlink는 필요 없다.

In [ ]:
import os

# cfg["data"]["root"]에 절대경로(DATA_ROOT)를 직접 넣으므로 symlink 불필요
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(CKPT_BACKUP, exist_ok=True)
print(f"CKPT_DIR    = {CKPT_DIR}")
print(f"CKPT_BACKUP = {CKPT_BACKUP}")

## 셀 10 - Colab용 config
`configs/baseline.yaml`을 읽어 Colab 환경에 맞게 덮어쓰고 `configs/colab_a1c1.yaml`로 저장한다.

In [ ]:
import yaml, torch, os

with open(f"{PROJECT_ROOT}/configs/baseline.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"]        = DATA_ROOT       # 절대경로 (symlink 없이 직접 가리킴)
cfg["data"]["num_workers"] = 2
cfg["data"]["pin_memory"]  = True
cfg["train"]["save_dir"]   = CKPT_DIR
cfg["train"]["epochs"]     = 15            # Free 12hr 한도 안에 맞추기 위해 줄임 (원본 20)

# Auto-detect protocol layout: tar 추출 결과가 .../protocols/ 폴더 안일 수도,
# DATA_ROOT 직속(flat)일 수도 있음. 실제 파일 위치 따라 경로 자동 조정.
def _find_tsv(name):
    for r, _, files in os.walk(DATA_ROOT):
        if name in files:
            rel = os.path.relpath(os.path.join(r, name), DATA_ROOT)
            return rel
    raise FileNotFoundError(f"{name} not under {DATA_ROOT}")

cfg["data"]["protocol_train"] = _find_tsv("ASVspoof5.train.tsv")
cfg["data"]["protocol_dev"]   = _find_tsv("ASVspoof5.dev.track_1.tsv")
try:
    cfg["data"]["protocol_eval"] = _find_tsv("ASVspoof5.eval.track_1.tsv")
except FileNotFoundError:
    pass  # eval은 선택 (아직 안 푼 경우)
print(f"protocol_train = {cfg['data']['protocol_train']}")
print(f"protocol_dev   = {cfg['data']['protocol_dev']}")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb < 14:
    cfg["train"]["batch_size"] = 4
elif vram_gb < 30:
    cfg["train"]["batch_size"] = 8
else:
    cfg["train"]["batch_size"] = 16
print(f"VRAM {vram_gb:.1f}GB → batch_size {cfg['train']['batch_size']}")

# A1×C1 명시
cfg["model"]["adapter"]["type"]    = "bilstm"
cfg["model"]["classifier"]["type"] = "lcnn_big"

OUT_CFG = f"{PROJECT_ROOT}/configs/colab_a1c1.yaml"
with open(OUT_CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f"wrote {OUT_CFG}")
print("\n--- key settings ---")
for k in ["batch_size", "epochs", "lr_peak", "early_stopping_patience"]:
    print(f"train.{k:<26}= {cfg['train'][k]}")
print(f"data.root                    = {cfg['data']['root']}")
print(f"model.ssl.name_or_path       = {cfg['model']['ssl']['name_or_path']}")
print(f"model.adapter.type           = {cfg['model']['adapter']['type']}")
print(f"model.classifier.type        = {cfg['model']['classifier']['type']}")
print(f"loss.lambda_workers          = {cfg['loss']['lambda_workers']}")

## 셀 11 - Smoke test (200 utterances × 2 epoch)
wiring / loss / gradient 확인용. loss가 줄고 dev EER < 40%면 통과.

In [ ]:
# smoke test는 wiring 검증용 -> W&B 대시보드에 남기지 않음(--no-wandb).
# 로컬 ckpt/smoke/train_log.jsonl · run_meta.json 은 그래도 생성됨.
!python train.py \
    --config configs/colab_a1c1.yaml \
    --tag smoke \
    --smoke 200 \
    --epochs 2 \
    --no-wandb

## 셀 12 - Sanity probe (선택)
adapter와 worker를 freeze하고 SSL layer weight + classifier만 300 step 학습한다.
dev EER 20–30% 정도가 나와야 하고, 훨씬 나쁘면 protocol 파싱이나 label 인코딩을 확인한다. 시간이 없으면 셀 13으로 넘어간다.

In [ ]:
import sys, yaml, numpy as np, torch
from sklearn.metrics import roc_curve

sys.path.insert(0, PROJECT_ROOT)
from data.dataset import build_dataloaders
from models.model import SSLMultiTaskModel

with open(f"{PROJECT_ROOT}/configs/colab_a1c1.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["train"]["batch_size"] = min(cfg["train"]["batch_size"], 8)

device = torch.device("cuda")
loaders = build_dataloaders(cfg)
model = SSLMultiTaskModel(cfg).to(device)

for p in model.adapter.parameters(): p.requires_grad = False
for p in model.workers.parameters(): p.requires_grad = False
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"sanity probe trainable params: {sum(p.numel() for p in trainable)/1e3:.1f}K")

opt = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
bce = torch.nn.BCEWithLogitsLoss()

# ~300 step
model.train()
for i, batch in enumerate(loaders["train"]):
    if i >= 300:
        break
    wav = batch["distorted"].to(device, non_blocking=True)
    lab = batch["label"].to(device).float()
    logit, _, _ = model(wav, apply_specaug=False)
    loss = bce(logit, lab)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if i % 50 == 0:
        print(f"step {i:3d}  loss {loss.item():.4f}")

model.eval()
scores, labels = [], []
with torch.no_grad():
    # dev에서 처음 5000 utterances만 빠르게 평가
    seen = 0
    for batch in loaders["dev"]:
        wav = batch["distorted"].to(device)
        logit, _, _ = model(wav, apply_specaug=False)
        scores.extend(torch.sigmoid(logit).cpu().numpy())
        labels.extend(batch["label"].numpy())
        seen += wav.shape[0]
        if seen >= 5000:
            break
scores = np.array(scores); labels = np.array(labels)
fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
fnr = 1.0 - tpr
idx = int(np.nanargmin(np.abs(fnr - fpr)))
eer = (fnr[idx] + fpr[idx]) / 2
print(f"\n[Sanity] dev (first 5K) EER: {eer*100:.2f}%  (expected 20–30%)")
if eer > 0.4:
    print("EER이 40%를 넘었습니다. protocol 파싱·label 인코딩 점검 필요.")

del model, opt, loaders
torch.cuda.empty_cache()

## 셀 13 - 본 학습 (A1×C1: BiLSTM + LCNN-big)
per-epoch dev는 5K 랜덤 subset, batch 32로 평가하고 train은 full 182K를 쓴다.
5분마다 ckpt를 Drive로 rsync한다. Drive에 `last.pt`가 있으면 `--resume`을 자동으로 붙여 다음 epoch부터 이어서 학습한다.

In [ ]:
import os, threading, time, subprocess

TAG = "a1c1"
src_dir = f"{CKPT_DIR}/{TAG}/"
dst_dir = f"{CKPT_BACKUP}/{TAG}/"
os.makedirs(src_dir, exist_ok=True)
os.makedirs(dst_dir, exist_ok=True)

# Resume 자동 감지
resume_args = []
backup_last = f"{CKPT_BACKUP}/{TAG}/last.pt"
local_last  = f"{CKPT_DIR}/{TAG}/last.pt"
if os.path.isfile(backup_last):
    if not os.path.isfile(local_last):
        subprocess.run(["cp", backup_last, local_last], check=True)
        # history.json도 함께 복원 (있으면)
        backup_hist = f"{CKPT_BACKUP}/{TAG}/history.json"
        local_hist  = f"{CKPT_DIR}/{TAG}/history.json"
        if os.path.isfile(backup_hist):
            subprocess.run(["cp", backup_hist, local_hist], check=False)
        print(f"[resume] copied last.pt (+history.json) from Drive backup")
    resume_args = ["--resume", local_last]
else:
    print(f"[resume] no prior last.pt - starting fresh")

# rsync 백그라운드 (5분마다 /content/ckpt -> Drive)
_stop = threading.Event()

def rsync_loop():
    while not _stop.is_set():
        for _ in range(60):
            if _stop.is_set():
                return
            time.sleep(5)
        try:
            r = subprocess.run(["rsync", "-a", "--delete", src_dir, dst_dir],
                               check=False, capture_output=True, text=True)
            if r.returncode == 0:
                print(f"[backup] {time.strftime('%H:%M:%S')}  rsynced → Drive")
        except Exception as e:
            print(f"[backup] error: {e}")

backup_thread = threading.Thread(target=rsync_loop, daemon=True)
backup_thread.start()
print("[backup] background rsync thread started (every 5 min)")

# train 실행
# 실험 추적: W&B에 로그인돼 있으면 --wandb로 실시간 기록, 미로그인/미설치면
# train.py가 자동으로 로컬 train_log.jsonl fallback (학습은 그대로 진행).
# W&B를 안 쓰려면 아래 리스트에서 "--wandb" 3줄을 "--no-wandb"로 바꾸면 됨.
#   (사전 로그인: 새 셀에서  !pip install -q wandb && python -c "import wandb; wandb.login()")
cmd = ["python", "train.py",
       "--config", "configs/colab_a1c1.yaml",
       "--tag", TAG,
       "--adapter", "bilstm",
       "--classifier", "lcnn_big",
       "--dev-batch-size", "32",
       "--dev-subset", "5000",
       "--wandb", "--wandb-project", "asvspoof5-track1", "--wandb-run-name", TAG] + resume_args
print("[train] cmd:", " ".join(cmd))

try:
    subprocess.run(cmd, check=False)
finally:
    _stop.set()
    subprocess.run(["rsync", "-a", "--delete", src_dir, dst_dir], check=False)
    print(f"\n[backup] final rsync → {dst_dir}")
    subprocess.run(["ls", "-la", dst_dir])

## 셀 14 - 학습 곡선 / WavLM layer weight

In [ ]:
import json, os, shutil, numpy as np, torch, matplotlib.pyplot as plt

TAG = "a1c1"

# 세션 재시작 후엔 CKPT_DIR이 비어있음 -> CKPT_BACKUP(Drive)에서 복원
local_dir = f"{CKPT_DIR}/{TAG}"
backup_dir = f"{CKPT_BACKUP}/{TAG}"
if not os.path.isfile(f"{local_dir}/best.pt"):
    if os.path.isfile(f"{backup_dir}/best.pt"):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(backup_dir, local_dir, dirs_exist_ok=True)
        print(f"[restore] {backup_dir} → {local_dir}")
    else:
        raise FileNotFoundError(f"best.pt 없음: {local_dir} 와 {backup_dir} 모두 비어있음")

with open(f"{local_dir}/history.json") as f:
    hist = json.load(f)

epochs     = [h["epoch"] for h in hist]
train_loss = [h["train"]["loss"]   for h in hist]
train_bce  = [h["train"]["bce"]    for h in hist]
train_w    = [h["train"]["worker"] for h in hist]
dev_eer    = [h["dev"]["eer"]*100  for h in hist]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, train_loss, label="total")
axes[0].plot(epochs, train_bce,  label="bce")
axes[0].plot(epochs, train_w,    label="worker")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("Train losses")

axes[1].plot(epochs, dev_eer, marker="o", color="crimson")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dev EER (%)")
axes[1].grid(alpha=0.3)
axes[1].set_title(f"dev EER  (best = {min(dev_eer):.2f}%)")
plt.tight_layout(); plt.show()

ckpt = torch.load(f"{local_dir}/best.pt", map_location="cpu")
lw = ckpt["model"]["ssl.layer_weights"]
softmaxed = torch.softmax(lw, dim=0).numpy()

plt.figure(figsize=(13, 3))
plt.bar(range(len(softmaxed)), softmaxed, color="steelblue")
plt.xlabel("WavLM layer index (0 = feature embedding, 24 = top)")
plt.ylabel("softmax weight")
plt.title("Learned WavLM layer importance")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

top3 = np.argsort(softmaxed)[-3:][::-1].tolist()
print(f"Best dev EER : {min(dev_eer):.2f}%  @ epoch {epochs[int(np.argmin(dev_eer))]}")
print(f"Best ckpt    : best_eer={ckpt['best_eer']*100:.2f}%, threshold={ckpt['best_threshold']:.4f}")
print(f"Top-3 layers : {top3}")

## 셀 15 - dev 평가 (official metric)
full dev(약 14만 발화)로 sliding window(4s/2s) 평가를 한 번 돌려 EER / minDCF / Cllr를 구한다. per-epoch 값은 5K subset 추정치였다.
score 파일 형식은 `<file_id> <score_bonafide> <label_int>`.

In [ ]:
import os, shutil

TAG = "a1c1"

# cwd 보장 (cell-8 건너뛰고 cell-28만 돌릴 때도 동작)
os.chdir(PROJECT_ROOT)

# 세션 재시작 후엔 CKPT_DIR이 비어있음 -> CKPT_BACKUP(Drive)에서 복원
local_dir = f"{CKPT_DIR}/{TAG}"
backup_dir = f"{CKPT_BACKUP}/{TAG}"
if not os.path.isfile(f"{local_dir}/best.pt"):
    if os.path.isfile(f"{backup_dir}/best.pt"):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(backup_dir, local_dir, dirs_exist_ok=True)
        print(f"[restore] {backup_dir} → {local_dir}")
    else:
        raise FileNotFoundError(f"best.pt 없음: {local_dir} 와 {backup_dir} 모두 비어있음")

!mkdir -p scores
!python evaluate.py \
    --config configs/colab_a1c1.yaml \
    --ckpt $CKPT_DIR/$TAG/best.pt \
    --split dev \
    --metric official \
    --sliding \
    --out scores/dev_${TAG}.tsv

!rsync -a scores/ $CKPT_BACKUP/$TAG/scores/
!rsync -a $CKPT_DIR/$TAG/ $CKPT_BACKUP/$TAG/
print(f"\n[backup] dev scores saved → {CKPT_BACKUP}/{TAG}/scores/")

## 참고

기대치 (실측 아님)

| 지표 | 기대값 |
|---|---|
| dev EER | 5–10% |
| dev minDCF | < 0.3 |

학습 곡선 볼 때
- `train.bce`는 감소, `train.worker`는 천천히 감소
- dev EER이 3–5 epoch에서 plateau면 early stopping이 걸린다
- WavLM layer weight가 중간층(8–16)에 몰리는지 확인

다른 조합은 셀 13의 `--adapter` / `--classifier`와 `TAG`를 바꿔서 돌리고, 셀 14–15의 `TAG`도 같이 바꾼다.

| 조합 | 셀 13 인자 |
|---|---|
| A1×C2 | `TAG="a1c2"`, `--adapter bilstm --classifier senet12` |
| A2×C1 | `TAG="a2c1"`, `--adapter se_res2net --classifier lcnn_big` |
| A2×C2 | `TAG="a2c2"`, `--adapter se_res2net --classifier senet12` |

eval set 평가
1. `python scripts/download_asvspoof5.py --local-dir $RAW_DIR --include-eval` (약 80 GB 추가)
2. 셀 8 재실행
3. 셀 15에서 `--split eval`로 변경

세션이 끊기면 셀 1–10을 다시 실행한 뒤 셀 13을 돌리면 Drive의 `last.pt`에서 이어진다. 평가만 할 때는 셀 13을 건너뛰고 셀 14–15를 실행한다 (ckpt는 `CKPT_BACKUP`에서 자동 복원).